In [2]:
import pandas as pd
import numpy as np

fy2016 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_Disclosure_Data_FY16.xlsx"

df = pd.read_excel(fy2016)

print(df.shape)
print(df.head())

(647852, 40)
          CASE_NUMBER          CASE_STATUS CASE_SUBMITTED DECISION_DATE  \
0  I-200-12240-490687  CERTIFIED-WITHDRAWN     2012-08-27    2015-10-16   
1  I-200-13053-847481  CERTIFIED-WITHDRAWN     2013-02-25    2016-01-13   
2  I-200-13088-054259  CERTIFIED-WITHDRAWN     2013-04-23    2015-12-10   
3  I-200-13144-034110  CERTIFIED-WITHDRAWN     2013-05-24    2016-01-19   
4  I-200-13172-415116            WITHDRAWN     2013-06-26    2016-05-20   

  VISA_CLASS EMPLOYMENT_START_DATE EMPLOYMENT_END_DATE  \
0       H-1B            2013-02-25          2016-02-24   
1       H-1B            2013-08-24          2016-08-23   
2       H-1B            2013-04-29          2016-04-28   
3       H-1B            2013-10-01          2016-09-30   
4       H-1B            2013-10-18          2016-10-17   

                                       EMPLOYER_NAME  \
0                             UNIVERSITY OF MICHIGAN   
1                             GOODMAN NETWORKS, INC.   
2                  

In [4]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'CASE_SUBMITTED', 'DECISION_DATE', 'VISA_CLASS', 'EMPLOYMENT_START_DATE', 'EMPLOYMENT_END_DATE', 'EMPLOYER_NAME', 'EMPLOYER_ADDRESS', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'AGENT_ATTORNEY_NAME', 'AGENT_ATTORNEY_CITY', 'AGENT_ATTORNEY_STATE', 'JOB_TITLE', 'SOC_CODE', 'SOC_NAME', 'NAIC_CODE', 'TOTAL_WORKERS', 'FULL_TIME_POSITION', 'PREVAILING_WAGE', 'PW_UNIT_OF_PAY', 'PW_WAGE_SOURCE', 'PW_SOURCE_YEAR', 'PW_SOURCE_OTHER', 'WAGE_RATE_OF_PAY_FROM', 'WAGE_RATE_OF_PAY_TO', 'WAGE_UNIT_OF_PAY', 'H-1B_DEPENDENT', 'WILLFUL_VIOLATOR', 'WORKSITE_CITY', 'WORKSITE_COUNTY', 'WORKSITE_STATE', 'WORKSITE_POSTAL_CODE', 'ORIGINAL_CERT_DATE']


In [28]:
import pandas as pd

df = pd.read_excel(fy2016)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
MICROSOFT CORPORATION                                                    5029
GOOGLE INC.                                                              4785
AMAZON CORPORATE LLC                                                     2547
APPLE INC.                                                               1656
FACEBOOK, INC.                                                           1167
TESLA MOTORS, INC.                                                        566
MICRON TECHNOLOGY, INC.                                                   361
AMAZON WEB SERVICES, INC.                                                 304
ADVANCED MICRO DEVICES, INC.                                              252
NVIDIA CORPORATION                                                        223
AMAZON.COM.DEDC, LLC                                                      104
APPLEXUS TECHNOLOGIES LLC                                                  94
GOOGLE, INC.                                      

In [33]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    5029
Apple        1656
Meta         1167
Tesla         566
Micron        361
AMD           252
NVIDIA        223
Name: count, dtype: int64


In [35]:
df_clean = df.copy()

In [37]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY"] = (
    df["WORKSITE_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      215
New York    118
Seattle     109
Dallas        9
Name: count, dtype: int64


In [39]:
print(df["WAGE_UNIT_OF_PAY"].value_counts())

WAGE_UNIT_OF_PAY
Year    451
Name: count, dtype: int64


In [41]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"]
    * df["WAGE_UNIT_OF_PAY"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]].head(10))

     Company      City  WAGE_RATE_OF_PAY_FROM WAGE_UNIT_OF_PAY  Annual_Wage
2716     AMD    Austin              107700.00             YEAR    107700.00
2812    Meta  New York              197160.06             YEAR    197160.06
2972    Meta  New York              175000.00             YEAR    175000.00
4161     AMD    Austin              118400.00             YEAR    118400.00
5294   Apple    Austin               90896.00             YEAR     90896.00
7012     AMD    Austin               86100.00             YEAR     86100.00
7086    Meta  New York              145410.56             YEAR    145410.56
7466    Meta   Seattle              181711.90             YEAR    181711.90
7915   Apple   Seattle              180000.00             YEAR    180000.00
7918    Meta  New York              196000.00             YEAR    196000.00


In [43]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count       451.000000
mean     120593.940865
std       34455.763462
min       41766.000000
25%       95000.000000
50%      115000.000000
75%      145000.000000
max      237534.940000
Name: Annual_Wage, dtype: float64


In [45]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [47]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                 89     94765.00   94913.078652
1     Austin      Apple                 92     92352.00   91892.293478
2     Austin       Meta                 10     99250.00  104302.810000
3     Austin     Micron                  4    150000.00  143076.750000
4     Austin  Microsoft                  4    143281.00  136685.500000
5     Austin     NVIDIA                 16    104562.00  104318.375000
6     Dallas      Apple                  6     98071.00  105536.666667
7     Dallas       Meta                  2    117595.47  117595.470000
8     Dallas  Microsoft                  1    105000.00  105000.000000
9   New York       Meta                 81    143010.70  145575.256543
10  New York  Microsoft                 37    135000.00  134164.783784
11   Seattle      Apple                  3    160000.00  158333.333333
12   Seattle       Meta                 94    150000.00  150052.473511
13   S

In [49]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_NAME",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2016.csv",
    index=False
)